# The mount: where the robot goes in the scene

The scene owns a **mount frame**; the task says where the robot goes *in that frame*. Two types:

- `Mount(offset=Pose(...))` for a fixed-base arm: `Mount()` snaps the base onto the frame, an offset shifts it in the frame's axes.
- `MobileMount(spawn=(x, y[, z]), heading=...)` for a robot with a floating base (a legged robot): where it spawns and which way it faces (radians, 0 = the frame's +x).

Until you write `mount=`, the builder picks the one that fits the robot. `world_mount(cfg)` is the robot base in the world.

In [1]:
import math

from robotenv.config import BaseEnvConfig, ConfigError, MobileMount, Mount, Pose, world_mount
from robotenv.robot.providers.menagerie import PROVIDER

go1 = PROVIDER.describe("unitree_go1")  # a floating-base model from MuJoCo Menagerie

## A fixed-base arm

The default mount snaps the arm onto the scene's frame. An offset moves it in the frame's axes.

In [2]:
arm = BaseEnvConfig()
print(arm.resolve().task.mount, "->", world_mount(arm.resolve()).pos)

shifted = arm.task(mount=Mount(offset=Pose((0.1, 0.0, 0.2), (1.0, 0.0, 0.0, 0.0))))
print(shifted.resolve().task.mount.offset.pos, "->", world_mount(shifted.resolve()).pos)

Mount(offset=Pose(pos=(0.0, 0.0, 0.0), quat=(1.0, 0.0, 0.0, 0.0))) -> (0.0, 0.0, 0.0)
(0.1, 0.0, 0.2) -> (0.1, 0.0, 0.2)


## A legged robot

No `mount=` needed: a floating-base robot gets `MobileMount()`, spawned at the frame facing +x.

In [3]:
go1_cfg = BaseEnvConfig().robot(go1).control(mode="joint_pos_delta", action_scale=0.05).sim(num_envs=2, scene="plain_floor")
go1_env = go1_cfg.build()
print(go1_env.cfg.task.mount)
print(go1_env.world_mount)

MobileMount(spawn=(0.0, 0.0, 0.0), heading=0.0)
Pose(pos=(0.0, 0.0, 0.0), quat=(1.0, 0.0, 0.0, 0.0))


Place it with a spawn and a heading. `z` is added to the robot's own standing height, so `z=0.3` drops it from 0.3 m.

In [4]:
placed = go1_cfg.task(mount=MobileMount(spawn=(1.0, 0.5, 0.0), heading=math.pi / 2))
print(world_mount(placed.resolve()))

Pose(pos=(1.0, 0.5, 0.0), quat=(0.7071067811865476, 0.0, 0.0, 0.7071067811865475))


## The types must fit the robot

A plain `Mount` on a floating-base robot is an error, and so is a `MobileMount` on a fixed-base one.

In [5]:
try:
    go1_cfg.task(mount=Mount()).resolve()
except ConfigError as e:
    print(e)

try:
    arm.task(mount=MobileMount()).resolve()
except ConfigError as e:
    print(e)

task.mount: robot 'unitree_go1' has a floating base: use MobileMount(spawn=(x, y), heading=...)
task.mount: MobileMount is for a robot with a floating base; a fixed-base robot takes Mount(offset=...)
